In [ ]:
!python --version

In [ ]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


### Defaults

In [ ]:
ROOT = Path().resolve().parent
root0 = ROOT / "data"

gdc = GDC(root0=root0)

os.listdir(root0)[:10]


### Get all programs

In [ ]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


In [ ]:
np.array(prog_list)

### Primary sites given a program

In [ ]:
gdc.url_gdc_project

In [ ]:
force=False
verbose=False

prog_id = 'TCGA'

gdc.set_program(prog_id)
df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)

i=0
primary_site = df_psi.iloc[i].primary_site
gdc.set_primary_site(primary_site=primary_site) 


In [ ]:
df_cases, df_all_samples, df_all_mut, barcode_list = gdc.get_filtered_tables(primary_site=primary_site, verbose=verbose)
len(df_all_mut), len(barcode_list)

In [ ]:
barcode_samples = np.unique(df_all_samples.barcode_sample)
len(barcode_samples)

In [ ]:
sample_types = np.unique(df_all_samples.sample_type)
sample_types

In [ ]:
cols = ['barcode', 'sample', 
       'symbol', 'refseq_mrna_id', 'entrez_gene_id', 'protein_mut',
       'mutation_type', 'ref_allele', 'variant_allele',
       'variant_type', 'chr', 'start', 'end', 'mutation_status',]

df_all_mut[cols].head()

### Mutations

In [ ]:
df_all_mut.columns

In [ ]:
df_all_mut.barcode.iloc[:4]

In [ ]:
barcode_list[:4]

In [ ]:
df4 = df_all_mut[df_all_mut.barcode_sample.isin(barcode_list)]
len(df4), len(df_all_mut)

### Mutations --> pivot: barcodes x genes

In [ ]:
dfpiv = gdc.build_pivot_table(df_all_mut)

dfpiv.head(3)

### UMAP cluster

#### Jaccard distance

Jaccard distance is a measure of dissimilarity between two sets, derived directly from Jaccard similarity. While Jaccard similarity measures how much two sets overlap, Jaccard distance measures how different they are. It is defined as one minus the Jaccard similarity.


$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$

In [ ]:
k = 4

embedding, labels = gdc.calc_UMAP(dfpiv, k)

df_umap = pd.DataFrame(
    embedding,
    index=dfpiv.index,
    columns=["UMAP1", "UMAP2"]
)

print(labels)
df_umap.head(6)

In [ ]:
selK = 0

barcodes = np.array(dfpiv.index.to_list())
selected_list = [True if x == selK else False for x in labels]

sel_barcodes = barcodes[selected_list]

print(sum(selected_list), len(sel_barcodes))
print("\n".join(sel_barcodes))

### My k-cluster 

### UMAP plot

In [ ]:
fig, embedding, labels = gdc.plot_umap(dfpiv, k=8)

In [ ]:
selK = 1

barcodes = np.array(dfpiv.index.to_list())
selected_list = [True if x == selK else False for x in labels]

sel_barcodes = barcodes[selected_list]

print(sum(selected_list), len(sel_barcodes))
print("\n".join(sel_barcodes))

In [ ]:
dfpiva = dfpiv[dfpiv.index.isin(sel_barcodes)]
print(dfpiva.shape)

dfpiva

In [ ]:
dfpiva_filt = dfpiva.loc[:, dfpiva.sum(axis=0) >= 2]
dfpiva_filt.shape
